# Imports and setup


In [ ]:
## Image viewing
import os
from pathlib import Path

import navis


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import colorcet as cc

IBMblue = "648FFF"
IBMpurple = "785EF0"
IBMhotpink = "DC267F"
IBMorange = "FE6100"
IBMyellow = "FFB000"

# Force PyImageJ to use Conda's Java, ignoring system defaults.
# Must be set BEFORE scyjava/imagej is imported (those trigger JVM startup).
if "CONDA_PREFIX" in os.environ:
    os.environ["JAVA_HOME"] = os.environ["CONDA_PREFIX"]

# Ensure the JVM is started in headless mode from the very first call.
# In a Jupyter kernel the JVM only starts once; if it comes up without the
# right options, the IJ1 legacy layer ends up 'Inactive' and IJ.openImage()
# (which the H5J_Loader_Plugin relies on) silently returns null.
import scyjava
scyjava.config.add_option("-Djava.awt.headless=true")

import imagej

# FIJI_APP = "/home/william-zheng/Downloads/Fiji.app"
# PROJECT_DIR = "/home/william-zheng/Documents/Programming/Python/NeuroInformatics/summer_2026/neuroinfo_fruitfly"
FIJI_APP = "/Users/vuhepola/Desktop/Fiji"
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FISBe_DIR = DATA_DIR / "FISBe"
FlyLight_DIR = DATA_DIR / "FlyLight"
FANC_DIR = DATA_DIR / "FANC"

print(FIJI_APP)
print(PROJECT_DIR)
print(DATA_DIR)
print(FISBe_DIR)
print(FlyLight_DIR)
print(FANC_DIR)

### Helper functions


In [ ]:
# Assuming 'scores' is your NBLAST result DataFrame
# Rows = Queries, Columns = Targets

def get_top_n_matches(nblast_scores, n=5):
    """
    Returns a DataFrame showing the top N matches for each query.
    """
    # Get the indices of the top N columns for each row
    # axis=1 sorts across columns
    top_indices = np.argsort(nblast_scores.values, axis=1)[:, ::-1][:, :n]
    
    # Create a new DataFrame for results
    top_matches = pd.DataFrame(index=nblast_scores.index)
    
    for i in range(n):
        # Add the Target ID
        top_matches[f'rank_{i+1}_id'] = nblast_scores.columns[top_indices[:, i]]
        # Add the corresponding NBLAST score
        top_matches[f'rank_{i+1}_score'] = nblast_scores.values[np.arange(nblast_scores.shape[0]), top_indices[:, i]]
        
    return top_matches

def get_k_colors(k, cmap_name='hsv'):
    # Fetch the colormap object
    cmap = plt.colormaps[cmap_name]
    
    # Linearly space k points between 0.0 and 1.0
    colors = [cmap(i) for i in np.linspace(0, 1, k)]
    return colors


# NAVis example skeletons


NAVis has built-in example neurons are from the [Janelia FlyEM team's hemibrain dataset](https://www.janelia.org/project-team/flyem/hemibrain)


In [ ]:
# Load a single example neuron
n = navis.example_neurons(n=1, kind='skeleton')
n

In [ ]:
n.to_swc("n.swc")

In [ ]:
n.nodes.head()

In [ ]:
navis.plot2d(n, view=('x', '-z'), color="coral", method='2d')

In [ ]:
navis.plot3d(n, color="coral")

In [ ]:
nl = navis.example_neurons(kind='skeleton')

In [ ]:
# Access the first neuron in the list
nl

In [ ]:
# Get the cable length for all neurons in the list
nl.cable_length

In [ ]:
# Generate a plot of our neurons
navis.plot2d(nl, view=('x', '-z'), method='2d')

In [ ]:
# Generate a plot of our neurons
navis.plot3d(nl)

## NBLAST on example neurons


In [ ]:
# Convert neurons into microns (they are 8nm)
nl_um = nl / (1000 / 8)
nl_um

query_n_um = nl_um[0]

In [ ]:
query_n_um_dp = navis.make_dotprops(query_n_um,k=4, resample=False)
nl_um_dp = navis.make_dotprops(nl_um,k=4, resample=False)

In [ ]:
scores = navis.nblast(query_n_um_dp, nl_um_dp)
print(scores)

In [ ]:
# Usage
top_k = get_top_n_matches(scores, n=2)
# print(top_5.head())
top_k
# Select a query and its top match
query_id = scores.index[0]
best_match_id = top_k.loc[query_id, 'rank_1_id']
next_best_match_id = top_k.loc[query_id, 'rank_2_id']
print("Best Match:",best_match_id)
print("2nd Best Match:",next_best_match_id)

# Get the actual neuron objects
# t_neuron = nl_um.idx[best_match_id]
t_neuron_2 = nl_um.idx[next_best_match_id]

# Visualize them using navis (Plotly backend)
import navis

# navis.plot3d([query_n_um,t_neuron,t_neuron_2], color=["purple",'blue','orange'], alpha=.5)
navis.plot3d([query_n_um,t_neuron_2], color=['lightblue','orange'], alpha=.5)

# plt.show()

**The direction of the comparison matters!**

Consider two very different neurons - one large, one small - that overlap in space. If the small neuron is the query, you will always find a close-by nearest-neighbour among the many points of the large target neuron. Consequently, this small
large comparison will produce a decent NBLAST score. By contrast, the other way around (large -> small) will likely produce a bad NBLAST score because many points in the large neuron are far away from the closest point in the small neuron. In practice, we typically use the mean between those forward and the reverse scores. This is done either by running two NBLASTs (query -> target and target -> query), or by passing e.g. scores="mean" to the respective NBLAST function.


In [ ]:
aba_scores = navis.nblast_allbyall(nl_um_dp)
# aba_mean

ax = sns.heatmap(aba_scores)
# ax.set(xlabel="", ylabel="", xticklabels="", yticklabels="")
plt.show()

In [ ]:
aba_mean = (aba_scores + aba_scores.T)/2

# aba_mean

ax = sns.heatmap(aba_mean)
# ax.set(xlabel="", ylabel="", xticklabels="", yticklabels="")
plt.show()

# NBLAST on FANC


## Load SWC files and types


In [ ]:
nt_identity = pd.read_excel(FANC_DIR / "nt_identity.xlsx")
nt_identity

In [ ]:
# Load a whole directory of target neurons into a NeuronList
target_neurons = navis.read_swc(FANC_DIR / "Skeletons" / "*.swc")

In [ ]:
target_neurons

In [ ]:
ordered_ids = nt_identity['pt_supervoxel_id'].astype(str).tolist()
print(len(ordered_ids))

In [ ]:
# 2. Build a list of the neurons in the exact order of your DataFrame
sorted_neurons_list = []
missing_ids = []

for pt_id in ordered_ids:
    # Search for the neuron whose name (filename) contains this sub-string ID
    # We check both .name and .id just to be safe
    match = next((n for n in target_neurons if pt_id in str(n.name) or pt_id in str(n.id)), None)
    
    if match is not None:
        sorted_neurons_list.append(match)
    else:
        missing_ids.append(pt_id)

# 3. Reconstruct into a new, sorted navis NeuronList
filtered_target_neurons = navis.NeuronList(sorted_neurons_list)

print(f"Successfully matched and sorted {len(filtered_target_neurons)} neurons.")
if missing_ids:
    print(f"Warning: {len(missing_ids)} IDs from your DataFrame were not found in the SWC folder.")

In [ ]:
# Create a set of the names of the neurons we actually kept
loaded_names_combined = " ".join([str(n.name) for n in filtered_target_neurons])

# Filter the DataFrame to rows where the pt_supervoxel_id exists inside our kept neurons
df_class_synced = nt_identity[nt_identity['pt_supervoxel_id'].astype(str).apply(lambda x: x in loaded_names_combined)].copy()

# Reset index so row 0 of df_class_synced perfectly matches filtered_target_neurons[0]
df_class_synced = df_class_synced.reset_index(drop=True)

df_class_synced["SWC filename"] = filtered_target_neurons.name

In [ ]:
filtered_target_neurons

In [ ]:
df_class_synced

In [ ]:
navis.plot3d(filtered_target_neurons)

## Generate a color map for every unique type in dataset


In [ ]:
unique_types = df_class_synced["classification_system"].unique()
palette = sns.color_palette("husl", len(unique_types))
color_map = {t: mcolors.to_hex(palette[i]) for i, t in enumerate(unique_types)}

In [ ]:
df_sorted = df_class_synced.sort_values(by="classification_system")

# 3. Use the sorted index permutation to rearrange the navis NeuronList
# df_sorted.index.values extracts the exact integer sequence needed to reorder the neurons
sorted_target_neurons = filtered_target_neurons[df_sorted.index.values]

# 4. Map the colors to match this new sorted order
sorted_neuron_colors = df_sorted["classification_system"].map(color_map).tolist()

# 5. CRITICAL: Reset the index of your sorted DataFrame 
# This keeps the DataFrame row indices (0, 1, 2...) permanently aligned with sorted_target_neurons
df_class_sorted = df_sorted.reset_index(drop=True)

# 6. Plot the sorted, beautifully grouped population
navis.plot3d(sorted_target_neurons, color=sorted_neuron_colors, alpha=.3)

In [ ]:
# k = df_class_synced["classification_system"].value_counts().shape[0]

# # 'husl' balances perceived brightness across hues
# colors = sns.color_palette("husl", k)

# print(colors) # Returns a list of RGBA tuples

In [ ]:
# filter = df_class_synced[df_class_synced["classification_system"] == "T1_L_Sensory"].index

# filter


# navis.plot3d(filtered_target_neurons[filter])

In [ ]:
for t in df_class_synced["classification_system"].value_counts().index[:10]:
    # Create a boolean filter mask for the current type
    sensory_mask = df_class_synced["classification_system"] == t
    temp_df = df_class_synced[sensory_mask]

    # Map using the global color_map (every neuron in this iteration will get t's global color)
    subset_colors = temp_df["classification_system"].map(color_map).tolist()

    # Plot this specific group
    print(f"Plotting group: {t} (Color: {color_map[t]})")
    navis.plot3d(filtered_target_neurons[sensory_mask.values], color=subset_colors, alpha=.5)

## Convert to dotprops


In [ ]:
query_neuron = sorted_target_neurons[0]

In [ ]:
# k=5 is the standard number of nearest neighbors used to compute the tangent vector
query_dp = navis.make_dotprops(query_neuron)
target_dp = navis.make_dotprops(sorted_target_neurons)

In [ ]:
query_dp

In [ ]:
target_dp[1]

## single neuron NBLAST


In [ ]:
scores = navis.nblast(query_dp, target_dp, normalized=True, n_cores=4)

print(scores)

In [ ]:
# Usage
top_5 = get_top_n_matches(scores, n=5)
# print(top_5.head())
top_5

In [ ]:
# Select a query and its top match
query_id = scores.index[0]
best_match_id = top_5.loc[query_id, 'rank_1_id']
next_best_match_id = top_5.loc[query_id, 'rank_2_id']
print("Best Match:",best_match_id)
print("2nd Best Match:",next_best_match_id)

# Get the actual neuron objects
t_neuron = sorted_target_neurons.idx[best_match_id]
t_neuron_2 = sorted_target_neurons.idx[next_best_match_id]

# Visualize them using navis (Plotly backend)
import navis

navis.plot3d([query_neuron,t_neuron,t_neuron_2], color=["purple",'blue','orange'], alpha=.5)

# plt.show()

## All-pairs NBLAST on FANC


In [ ]:
score_matrix = navis.nblast_allbyall(target_dp)

In [ ]:
ax = sns.heatmap(score_matrix)
ax.set(xlabel="", ylabel="", xticklabels="", yticklabels="")
plt.show()

### All-pairs Mean NBLAST


In [ ]:
score_matrix_mean = (score_matrix + score_matrix.T)/2

In [ ]:
ax = sns.heatmap(score_matrix_mean)
# ax.set(xlabel="", ylabel="", xticklabels="", yticklabels="")
plt.show()

# NBLAST on MCFO


In [ ]:
print("Initializing headless Fiji environment...")
ij = imagej.init(FIJI_APP, mode="headless", add_legacy=True)
# ij = imagej.init()
# ij.getVersion() returns a java.lang.String; coerce to a Python str.
version = str(ij.getVersion())
print(f"ImageJ version: {version}")
if version.endswith("/Inactive"):
    raise RuntimeError(
        "Legacy ImageJ1 is Inactive in this kernel. Restart the Jupyter "
        "kernel (Kernel -> Restart) and rerun this cell from a clean state."
    )
    
IJ = scyjava.jimport("ij.IJ")

In [ ]:
# Launch ImageJ in headless mode using your defined variables

# Path to your target h5j file
# h5j_file_path =  FlyLight_DIR / "SS04748/SS04748-20170324_20_I1-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j"
npz_file_path =  FlyLight_DIR / "SS04748/SS04748-20170324_20_I1.npz"

# Load via Fiji's legacy layer
# imp = IJ.openImage(str(h5j_file_path))

# # Convert the ImagePlus object to a NumPy array
# # Shape will typically be (Channels, Z, Y, X) or (Z, Channels, Y, X)
# img_array = ij.py.from_java(imp)
with np.load(npz_file_path) as data:
    img_array = data['image_data']
    print("Image Shape:", img_array.shape)

In [ ]:
import scipy.ndimage as ndimage
from skimage.filters import threshold_otsu

# 1. Isolate your channel (e.g., Channel 0)
# Adjust slice indexing depending on your stack's shape layout
channel_data = img_array[:, 0, :, :]

print(channel_data.shape)

channel_data = np.transpose(channel_data, (1,2,0))

print(channel_data.shape)

# 2. De-noising (Crucial for H.265 compression artifacts)
# filtered_data = ndimage.gaussian_filter(channel_data, sigma=1.0)

# 3. Thresholding to create a binary mask (True = Neuron, False = Background)
thresh = threshold_otsu(channel_data)
binary_mask = channel_data > thresh

In [ ]:
# Note: You must provide the true voxel dimensions (microns or nm per pixel) 
# voxel_dims = (0.52, 0.52,1.0) # Example scaling: (x, y, z) sizes in microns

voxel_neuron = navis.VoxelNeuron(channel_data)

# Optional: View the raw voxels to ensure spacing is correct
navis.plot3d([voxel_neuron])

In [ ]:
import numpy as np
import navis
import trimesh
import skeletor
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.measure import marching_cubes

# 1. Standard Thresholding
thresh = threshold_otsu(channel_data)
binary_mask = channel_data > thresh

# 2. Label distinct connected 3D objects
# background=0 treats 0 as the empty space
labeled_mask, num_features = label(binary_mask, background=0, return_num=True)
print(f"Found {num_features} distinct components (including noise).")

# 3. Analyze the components to filter out small noise fragments
props = regionprops(labeled_mask)

# Sort components by size (volume) in descending order
# x.area in 3D gives the total voxel count of that component
sorted_props = sorted(props, key=lambda x: x.area, reverse=True)

# 4. Extract individual masks for the top largest objects
# Let's say you want to isolate the 2 largest distinct neurons
neuron_masks = []
for i in range(min(2, len(sorted_props))): # Adjust '2' to how many main neurons you expect
    neuron_id = sorted_props[i].label
    neuron_size = sorted_props[i].area
    print(f"Neuron {i+1} has an ID of {neuron_id} and size of {neuron_size} voxels.")
    
    # Create a clean binary mask containing ONLY this specific neuron
    individual_mask = (labeled_mask == neuron_id)
    neuron_masks.append(individual_mask)

# 5. Process one of the isolated neurons through your existing skeletonization pipeline
# (Change index to 1 for the second neuron, etc.)
chosen_neuron_mask = neuron_masks[0] 

voxel_dims = (0.52, 0.52, 1.0) 
verts, faces, normals, values = marching_cubes(chosen_neuron_mask, spacing=voxel_dims)
neuron_mesh = trimesh.Trimesh(vertices=verts, faces=faces)
skel_obj = skeletor.skeletonize.by_wavefront(neuron_mesh)
query_neuron = navis.TreeNeuron(skel_obj.swc)

navis.plot3d([query_neuron], color =["blue"])

In [ ]:
# Convert to dotprops (resample every 1 micron)
query_dp = navis.make_dotprops(query_neuron)
target_dp = navis.make_dotprops(sorted_target_neurons)

# Run NBLAST
scores = navis.nblast(query_dp, target_dp, n_cores = 8, normalized=True, scores="mean")

In [ ]:
scores

In [ ]:
# Usage
top_5 = get_top_n_matches(scores, n=5)
# print(top_5.head())
top_5

In [ ]:
# Select a query and its top match
query_id = scores.index[0]
best_match_id = top_5.loc[query_id, 'rank_1_id']
next_best_match_id = top_5.loc[query_id, 'rank_2_id']
print("Best Match:",best_match_id)
print("2nd Best Match:",next_best_match_id)

# Get the actual neuron objects
t_neuron = target_neurons.idx[best_match_id]
t_neuron_2 = target_neurons.idx[next_best_match_id]

# Visualize them using navis (Plotly backend)
import navis

navis.plot3d([query_neuron,t_neuron,t_neuron_2], color=['blue','orange', 'purple'], alpha=.5)

# plt.show()